# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

This completed version uses the packaged `Class6/database` and `Class6/data` files.


In [1]:
import json
import sqlite3
from pathlib import Path
import pandas as pd

BASE = Path.cwd()
if not (BASE / "database" / "flasheats.db").exists():
    # Colab / notebook-directory fallback
    for candidate in [Path("/content/Class6"), Path("/content"), Path.cwd()]:
        if (candidate / "database" / "flasheats.db").exists():
            BASE = candidate
            break

DB_PATH = BASE / "database" / "flasheats.db"
con = sqlite3.connect(DB_PATH)
orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)
tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")
with open(BASE / "data" / "client_metric_definitions.json") as f:
    metric_notes = json.load(f)

print("BASE:", BASE)
print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

BASE: /mnt/data/fix/FlashEats_Class6_Class8_Completed/Class6
orders: (1603, 13)
tickets: (177, 5)
restaurant_status: (1603, 4)


## Load the updated FlashEats data

In [2]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13)
tickets: (177, 5)
restaurant_status: (1603, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,None,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,category,created_at,status
0,T-0001,O00037,Late Delivery,2026-08-09T21:09:00,open
1,T-0002,O00040,Late_Delivery,2026-08-11T20:15:00,closed
2,T-0003,O00041,ETA Issue,2026-08-01T12:24:00,closed
3,T-0004,O00042,restaurant_delay,2026-08-09T19:19:00,closed
4,T-0005,O00049,Late_Delivery,2026-08-09T11:42:00,closed


,order_id,restaurant_id,status,last_updated_at
0,O00001,R009,ready,2026-08-14T13:35:49.825561
1,O00002,R018,preparing,2026-08-01T19:06:24.735336
2,O00003,R010,delayed,2026-08-01T20:08:50.820366
3,O00004,R030,handoff,2026-08-09T19:00:48.725485
4,O00005,R004,handed_off,2026-08-06T21:35:38.024280


{
  "definitions": [
    {
      "name": "any_delay_positive",
      "description": "Late when actual delivery is more than 0 minutes after promised ETA; denominator is delivered rows with both timestamps present."
    },
    {
      "name": "delay_over_10_minutes",
      "description": "Late when actual delivery is more than 10 minutes after promised ETA; denominator is delivered rows with both timestamps present."
    },
    {
      "name": "historical_delivered_non_null",
      "description": "Historical-style calculation using delivered rows with non-null promised ETA and actual delivery timestamps; late means positive delay."
    }
  ],
  "ownership": "The KPI/business owner must approve the published definition and denominator before leadership reporting."
}


# Challenge 1 — Can we defend the “56% late” claim?


Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order |  |  |  |
| Delivered orders have completion time |  |  |  |
| Promised ETA is valid |  |  |  |
| Event chronology is valid |  |  |  |
| “Late” has an agreed definition |  |  |  |

**Hint:** Don't start by cleaning. Ask: *what could make 56% misleading?*

In [3]:
print("Rows:", len(orders))
print("Unique orders:", orders["order_id"].nunique())
print(orders["final_status"].value_counts(dropna=False))

checks = {}
checks["business_grain"] = {"status": "WARN", "evidence": f"{len(orders)} source rows vs {orders.order_id.nunique()} unique orders; duplicate source rows exist."}
normalized = orders["final_status"].astype(str).str.strip().str.lower()
delivered = normalized.eq("delivered")
checks["delivered_completion"] = {"status": "WARN", "evidence": f"{int((delivered & orders.actual_delivery_at.isna()).sum())} delivered rows have missing actual_delivery_at."}
for c in ["created_at","promised_eta","pickup_at","actual_delivery_at"]:
    orders[c] = pd.to_datetime(orders[c], format="mixed", errors="coerce")
checks["promised_eta"] = {"status": "FAIL", "evidence": f"{int((orders.promised_eta < orders.created_at).sum())} rows have promised_eta before created_at."}
checks["chronology"] = {"status": "FAIL", "evidence": f"{int((orders.pickup_at > orders.actual_delivery_at).sum())} rows have pickup_at after actual_delivery_at."}
checks["late_definition"] = {"status": "UNKNOWN", "evidence": "The source does not establish an approved lateness threshold/denominator; metric ownership is required."}

validation_contract = pd.DataFrame([
    ["One row = one business order", "order_id unique at business grain", "Count rows vs nunique(order_id)", "WARN"],
    ["Delivered orders have completion time", "Delivered implies actual_delivery_at present", "Count missing completion timestamps", "WARN"],
    ["Promised ETA is valid", "promised_eta >= created_at", "Compare parsed timestamps", "FAIL"],
    ["Event chronology is valid", "pickup_at <= actual_delivery_at", "Compare lifecycle timestamps", "FAIL"],
    ["Late has an agreed definition", "Approved threshold + denominator", "Inspect metric definition and owner", "UNKNOWN"],
], columns=["Business assumption","Data expectation","How tested","Severity/status"])
display(validation_contract)

Rows: 1603
Unique orders: 1600
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64


,Business assumption,Data expectation,How tested,Severity/status
0,One row = one business order,order_id unique at business grain,Count rows vs nunique(order_id),WARN
1,Delivered orders have completion time,Delivered implies actual_delivery_at present,Count missing completion timestamps,WARN
2,Promised ETA is valid,promised_eta >= created_at,Compare parsed timestamps,FAIL
3,Event chronology is valid,pickup_at <= actual_delivery_at,Compare lifecycle timestamps,FAIL
4,Late has an agreed definition,Approved threshold + denominator,Inspect metric definition and owner,UNKNOWN


# Challenge 2 — Stakeholders disagree on “late”


Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|

Then answer:

> **Which one should leadership publish, and who must own that decision?**

In [4]:
analysis = orders.drop_duplicates("order_id", keep="first").copy()
for c in ["promised_eta", "actual_delivery_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")
analysis["final_status_norm"] = analysis["final_status"].astype(str).str.strip().str.lower()
analysis["delay_min"] = (analysis["actual_delivery_at"] - analysis["promised_eta"]).dt.total_seconds() / 60
measurable = analysis[(analysis.final_status_norm == "delivered") & analysis.actual_delivery_at.notna() & analysis.promised_eta.notna()].copy()
rows = [
    ["> 0 minutes", round((measurable.delay_min > 0).mean()*100,2), f"Positive delay among {len(measurable)} measurable delivered orders."],
    ["> 10 minutes", round((measurable.delay_min > 10).mean()*100,2), f"Material delay above a 10-minute threshold among {len(measurable)} measurable delivered orders."],
    ["Historical delivered/non-null", round((measurable.delay_min > 0).mean()*100,2), "Same measurable delivered/non-null population, with positive delay as the late condition."],
]
metric_comparison = pd.DataFrame(rows, columns=["Definition","Late rate (%)","Business meaning"])
display(metric_comparison)
print("Decision ownership: the KPI/business owner must approve the definition and denominator before publication; the data team should not silently choose one.")

,Definition,Late rate (%),Business meaning
0,> 0 minutes,56.39,Positive delay among 1495 measurable delivered...
1,> 10 minutes,23.34,Material delay above a 10-minute threshold amo...
2,Historical delivered/non-null,56.39,"Same measurable delivered/non-null population,..."


Decision ownership: the KPI/business owner must approve the definition and denominator before publication; the data team should not silently choose one.


# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

Remember:

> `"Delivered"` vs `"delivered"` is likely representation.  
> `"handoff"` vs `"handed_off"` may be semantics.

In [5]:
category_results = {}
for col in ["final_status", "traffic_bucket"]:
    category_results[col] = orders[col].value_counts(dropna=False).to_dict()
category_results["restaurant_status"] = restaurant_status["status"].value_counts(dropna=False).to_dict()
category_results["support_category"] = tickets["category"].value_counts(dropna=False).to_dict()

for k, v in category_results.items():
    print("\n", k, v)

print("\nColumn-by-column assessment:")
print("- final_status: casing/whitespace variants are representation differences and are safe to normalize after confirming the vocabulary; lifecycle states themselves must not be merged by instinct.")
print("- traffic_bucket: casing/whitespace variants such as HIGH/high are representation differences; normalize representation only after confirming the canonical vocabulary.")
print("- restaurant status: values such as ready, preparing, delayed, handoff and handed_off should not be merged automatically; handoff vs handed_off may represent different lifecycle states and requires owner confirmation.")
print("- support-ticket category: categories should be treated as business labels; only obvious casing/whitespace representation differences are safe to normalize without changing meaning.")
print("\nRule: normalize representation differences; obtain owner confirmation before normalizing semantic differences.")



 final_status {'delivered': 1530, 'cancelled': 68, 'Delivered': 5}

 traffic_bucket {'medium': 635, 'high': 448, 'low': 381, 'severe': 136, 'HIGH': 3}

 restaurant_status {'ready': 321, 'preparing': 321, 'delayed': 321, 'handoff': 320, 'handed_off': 320}

 support_category {'restaurant_delay': 34, 'Late_Delivery': 33, 'ready_but_waiting': 31, 'ETA_CHANGE': 30, 'Status_Mismatch': 26, 'driver_not_moving': 21, 'ETA Issue': 1, 'Late Delivery': 1}

Column-by-column assessment:
- final_status: casing/whitespace variants are representation differences and are safe to normalize after confirming the vocabulary; lifecycle states themselves must not be merged by instinct.
- traffic_bucket: casing/whitespace variants such as HIGH/high are representation differences; normalize representation only after confirming the canonical vocabulary.
- restaurant status: values such as ready, preparing, delayed, handoff and handed_off should not be merged automatically; handoff vs handed_off may represent dif

# Challenge 4 — Cross-source integrity


Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

Produce:

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|

Then discuss:

> If 1% is unmapped, is that acceptable?

It depends on which business decision uses those records.

In [6]:
def coverage(left, right, key):
    vals = left[key].dropna()
    return (vals.isin(set(right[key].dropna())).mean()*100) if len(vals) else 100.0

rows=[]
for name,left,right,key in [
    ("orders.restaurant_id → restaurants", orders, restaurants, "restaurant_id"),
    ("orders.driver_id → drivers", orders, drivers, "driver_id"),
    ("tickets.order_id → orders", tickets, orders, "order_id"),
    ("restaurant_status.order_id → orders", restaurant_status, orders, "order_id"),
]:
    cov=coverage(left,right,key)
    rows.append([name,round(cov,2),"PASS" if cov==100 else "WARN", "Assess impact by business decision; do not assume 1% is acceptable."])
mapping_report=pd.DataFrame(rows,columns=["Relationship","Coverage %","Status","Risk"])
display(mapping_report)
print("A 1% unmapped rate is not automatically acceptable or unacceptable. The owner must determine whether those records affect the decision being supported.")


,Relationship,Coverage %,Status,Risk
0,orders.restaurant_id → restaurants,100.00,PASS,Assess impact by business decision; do not ass...
1,orders.driver_id → drivers,100.00,PASS,Assess impact by business decision; do not ass...
2,tickets.order_id → orders,100.00,PASS,Assess impact by business decision; do not ass...
3,restaurant_status.order_id → orders,99.38,WARN,Assess impact by business decision; do not ass...


A 1% unmapped rate is not automatically acceptable or unacceptable. The owner must determine whether those records affect the decision being supported.


# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

In [7]:
status = restaurant_status.copy()
o = orders[["order_id","created_at","pickup_at","actual_delivery_at"]].copy()
for c in ["created_at","pickup_at","actual_delivery_at"]:
    o[c]=pd.to_datetime(o[c],format="mixed",errors="coerce")
status["last_updated_at"]=pd.to_datetime(status["last_updated_at"],format="mixed",errors="coerce")
joined=status.merge(o,on="order_id",how="left")
reference=joined["pickup_at"].fillna(joined["created_at"])
joined["update_lag_min"]=(joined["last_updated_at"]-reference).dt.total_seconds()/60
print(joined["update_lag_min"].describe(percentiles=[.5,.95,.99]))
print("Observed freshness must be evaluated against explicit SLA thresholds. Without approved thresholds, the validation status is UNKNOWN for live ETA/accountability use, while the observed lag can still be reported for weekly analytics.")

count    1596.000000
mean       29.286967
std        95.538712
min         5.000000
50%        23.000000
95%        38.000000
99%       180.000000
max      1440.000000
Name: update_lag_min, dtype: float64
Observed freshness must be evaluated against explicit SLA thresholds. Without approved thresholds, the validation status is UNKNOWN for live ETA/accountability use, while the observed lag can still be reported for weekly analytics.


# Challenge 6 — Build the validation gate


Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain |  |  |  |
| Timestamp chronology |  |  |  |
| KPI definition |  |  |  |
| Category semantics |  |  |  |
| Cross-source mapping |  |  |  |
| Freshness |  |  |  |

Use only:
**PASS / WARN / FAIL / UNKNOWN**

Finally answer:

> **Should leadership publish “Late Delivery Rate = 56%” today?**

What exactly must happen before the metric becomes publishable?

In [8]:
# Build the final gate from the values calculated above so the evidence cannot drift from the packaged data.
duplicate_rows = len(orders) - orders["order_id"].nunique()
promised_before_created = int((orders["promised_eta"] < orders["created_at"]).sum())
pickup_after_delivery = int((orders["pickup_at"] > orders["actual_delivery_at"]).sum())
delivered_missing_actual = int((orders["final_status"].astype(str).str.strip().str.lower().eq("delivered") & orders["actual_delivery_at"].isna()).sum())

status_cov = mapping_report.loc[mapping_report["Relationship"] == "restaurant_status.order_id → orders", "Coverage %"].iloc[0]

validation_report = {
    "business_grain": {"status":"WARN", "evidence":f"{len(orders)} source rows vs {orders.order_id.nunique()} unique orders; {duplicate_rows} duplicate source rows exist.", "action":"Retain one row per agreed order grain and monitor source duplicates."},
    "timestamp_chronology": {"status":"FAIL" if (promised_before_created or pickup_after_delivery) else "PASS", "evidence":f"{promised_before_created} promised_eta values precede created_at and {pickup_after_delivery} pickup_at values follow actual_delivery_at.", "action":"Investigate source timestamp semantics/data defects before publishing lifecycle KPIs."},
    "kpi_definition": {"status":"UNKNOWN", "evidence":f"Positive-delay measurable population gives {metric_comparison.loc[metric_comparison['Definition']=='> 0 minutes','Late rate (%)'].iloc[0]:.2f}%, but ownership of threshold/denominator is not established.", "action":"Obtain KPI owner approval for the published definition."},
    "category_semantics": {"status":"WARN", "evidence":"Representation differences can be normalized, but lifecycle labels such as handoff vs handed_off may be semantic.", "action":"Normalize only confirmed representations; obtain owner mapping for semantic variants."},
    "cross_source_mapping": {"status":"WARN" if (mapping_report["Coverage %"] < 100).any() else "PASS", "evidence":f"restaurant_status→orders coverage is {status_cov:.2f}%; other relationships are shown in the mapping report.", "action":"Assess unmapped records by decision impact and resolve/approve tolerance."},
    "freshness": {"status":"UNKNOWN", "evidence":"Observed update lag was measured, but approved SLA thresholds for weekly analytics, live ETA and accountability are not supplied.", "action":"Obtain explicit freshness SLAs for each use case and compare the observed lag against them."},
}

validation_table = pd.DataFrame([
    [k, v["status"], v["evidence"], v["action"]] for k,v in validation_report.items()
], columns=["Check","Status","Evidence","Action"])
display(validation_table)
print("Final decision: do not treat 56% as an approved leadership KPI until the FAIL/UNKNOWN items are resolved and the KPI owner approves the definition.")


,Check,Status,Evidence,Action
0,business_grain,WARN,1603 source rows vs 1600 unique orders; 3 dupl...,Retain one row per agreed order grain and moni...
1,timestamp_chronology,FAIL,4 promised_eta values precede created_at and 5...,Investigate source timestamp semantics/data de...
2,kpi_definition,UNKNOWN,Positive-delay measurable population gives 56....,Obtain KPI owner approval for the published de...
3,category_semantics,WARN,"Representation differences can be normalized, ...",Normalize only confirmed representations; obta...
4,cross_source_mapping,WARN,restaurant_status→orders coverage is 99.38%; o...,Assess unmapped records by decision impact and...
5,freshness,UNKNOWN,"Observed update lag was measured, but approved...",Obtain explicit freshness SLAs for each use ca...


Final decision: do not treat 56% as an approved leadership KPI until the FAIL/UNKNOWN items are resolved and the KPI owner approves the definition.


# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**

For this dataset, the 56% claim is reproducible as **56.39%** when late means `actual_delivery_at - promised_eta > 0` minutes over delivered orders with both timestamps present. However, timestamp chronology defects, duplicate source rows, category semantics, mapping exceptions, freshness SLAs, and KPI ownership prevent treating that number as an approved leadership metric without further action.